In [ ]:
# ── CELL 1: Mount + imports ──────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, random, numpy as np
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from collections import defaultdict
import matplotlib.pyplot as plt
import matplotlib.patches as patches

NAMES = {0:'person', 1:'car', 2:'motorcycle', 3:'truck', 4:'traffic light', 5:'stop sign'}
COLORS = {0:'#FF4444', 1:'#44FF44', 2:'#4444FF', 3:'#FF8800', 4:'#FF44FF', 5:'#44FFFF'}
IMG_EXTS = {'.jpg','.jpeg','.png','.bmp'}

# ── All dataset roots ─────────────────────────────────────────
ORIG_ROOT    = '/content/drive/MyDrive/2dod/2dod'       # original dataset
CLEAN_640    = '/content/drive/MyDrive/Patched Datasets/Clean_640'
PATCHED_ROOT = '/content/drive/MyDrive/Patched Datasets'

print('Paths set ✅')

In [ ]:
# ── CELL 2: Core helpers ─────────────────────────────────────
def read_labels(label_path):
    """Read YOLO labels. Returns list of (cls, cx, cy, w, h) normalised."""
    p = Path(label_path)
    if not p.exists(): return []
    out = []
    for ln in p.read_text(errors='ignore').strip().splitlines():
        parts = ln.strip().split()
        if len(parts) < 5: continue
        c = int(float(parts[0]))
        cx,cy,bw,bh = map(float, parts[1:5])
        out.append((c, cx, cy, bw, bh))
    return out

def check_label(c, cx, cy, bw, bh):
    """Returns list of issues with this label."""
    issues = []
    if c not in NAMES: issues.append(f'unknown_class_{c}')
    if not (0 <= cx <= 1): issues.append(f'cx_out_of_range:{cx:.3f}')
    if not (0 <= cy <= 1): issues.append(f'cy_out_of_range:{cy:.3f}')
    if not (0 < bw <= 1):  issues.append(f'bw_invalid:{bw:.3f}')
    if not (0 < bh <= 1):  issues.append(f'bh_invalid:{bh:.3f}')
    if bw > 0.95 and bh > 0.95: issues.append('box_covers_full_image')
    return issues

def scan_folder(img_dir, lab_dir, name=''):
    """Scan a folder pair and return stats dict."""
    img_dir, lab_dir = Path(img_dir), Path(lab_dir)
    imgs = sorted([f for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXTS])
    if not imgs: return None

    stats = {
        'name': name, 'n_images': len(imgs),
        'n_labels_found': 0, 'n_labels_missing': 0,
        'n_empty_labels': 0, 'n_boxes_total': 0,
        'class_counts': defaultdict(int),
        'issues': [], 'bad_images': [],
        'boxes_per_image': [],
    }

    for img_path in imgs:
        lbl_path = lab_dir / f'{img_path.stem}.txt'
        if not lbl_path.exists():
            stats['n_labels_missing'] += 1
            continue
        stats['n_labels_found'] += 1
        labels = read_labels(lbl_path)
        if not labels:
            stats['n_empty_labels'] += 1
        stats['n_boxes_total'] += len(labels)
        stats['boxes_per_image'].append(len(labels))
        for (c,cx,cy,bw,bh) in labels:
            stats['class_counts'][c] += 1
            bad = check_label(c,cx,cy,bw,bh)
            if bad:
                stats['issues'].append((str(img_path.name), c, bad))
                if img_path.name not in stats['bad_images']:
                    stats['bad_images'].append(img_path.name)

    return stats

def print_stats(s):
    if s is None: print('  [empty folder]'); return
    avg = np.mean(s['boxes_per_image']) if s['boxes_per_image'] else 0
    print(f"  Images       : {s['n_images']}")
    print(f"  Labels found : {s['n_labels_found']}  missing: {s['n_labels_missing']}")
    print(f"  Empty labels : {s['n_empty_labels']}")
    print(f"  Total boxes  : {s['n_boxes_total']}  avg/img: {avg:.2f}")
    print(f"  Class dist   : { {NAMES.get(c,c): v for c,v in sorted(s['class_counts'].items())} }")
    if s['issues']:
        print(f"  ⚠️  Issues    : {len(s['issues'])} bad boxes in {len(s['bad_images'])} images")
        for img, c, bad in s['issues'][:5]:
            print(f"      {img}: cls={c} → {bad}")
        if len(s['issues']) > 5:
            print(f"      ... and {len(s['issues'])-5} more")
    else:
        print(f"  ✅ No label issues")

print('Helpers ready ✅')

In [ ]:
# ── CELL 3: Scan ORIGINAL dataset (2dod/2dod) ────────────────
print('='*70)
print('ORIGINAL DATASET: 2dod/2dod')
print('='*70)

orig_stats = {}
for bb_num in range(1, 10):
    bb = f'billboard0{bb_num}'
    for split in ['train', 'val', 'test_nopatch', 'test_random']:
        img_dir = Path(ORIG_ROOT)/bb/f'images/{split}'
        lab_dir = Path(ORIG_ROOT)/bb/f'labels/{split}'
        if not img_dir.exists(): continue
        key  = f'{bb}/{split}'
        s    = scan_folder(img_dir, lab_dir, key)
        orig_stats[key] = s
        print(f'\n── {key} ──')
        print_stats(s)

# Overall summary
total_imgs  = sum(s['n_images']      for s in orig_stats.values() if s)
total_boxes = sum(s['n_boxes_total'] for s in orig_stats.values() if s)
total_bad   = sum(len(s['issues'])   for s in orig_stats.values() if s)
print(f'\n{"="*70}')
print(f'ORIGINAL TOTAL: {total_imgs} images, {total_boxes} boxes, {total_bad} bad labels')

In [ ]:
# ── CELL 4: Scan Clean_640 ────────────────────────────────────
print('='*70)
print('CLEAN_640 DATASET')
print('='*70)

clean_stats = {}
for bb_num in range(1, 10):
    img_dir = Path(CLEAN_640)/f'images/billboard0{bb_num}'
    lab_dir = Path(CLEAN_640)/f'labels/billboard0{bb_num}'
    if not img_dir.exists(): continue
    key = f'billboard0{bb_num}'
    s   = scan_folder(img_dir, lab_dir, key)
    clean_stats[key] = s
    print(f'\n── {key} ──')
    print_stats(s)

total_imgs  = sum(s['n_images']      for s in clean_stats.values() if s)
total_boxes = sum(s['n_boxes_total'] for s in clean_stats.values() if s)
total_bad   = sum(len(s['issues'])   for s in clean_stats.values() if s)
print(f'\n{"="*70}')
print(f'CLEAN_640 TOTAL: {total_imgs} images, {total_boxes} boxes, {total_bad} bad labels')

In [ ]:
# ── CELL 5: Scan Patched datasets ────────────────────────────
print('='*70)
print('PATCHED DATASETS')
print('='*70)

patch_stats = {}
for bb_num in range(1, 10):
    img_dir = Path(PATCHED_ROOT)/f'patched_dataset{bb_num}/images/test'
    lab_dir = Path(CLEAN_640)/f'labels/billboard0{bb_num}'  # GT from Clean_640
    if not img_dir.exists(): continue
    key = f'patched_dataset{bb_num}'
    s   = scan_folder(img_dir, lab_dir, key)
    patch_stats[key] = s
    print(f'\n── {key} ──')
    print_stats(s)

total_imgs  = sum(s['n_images']      for s in patch_stats.values() if s)
total_boxes = sum(s['n_boxes_total'] for s in patch_stats.values() if s)
total_bad   = sum(len(s['issues'])   for s in patch_stats.values() if s)
print(f'\n{"="*70}')
print(f'PATCHED TOTAL: {total_imgs} images, {total_boxes} boxes, {total_bad} bad labels')

In [ ]:
# ── CELL 6: Compare Clean_640 labels vs Original labels ───────
# This is the key diagnostic — are Clean_640 labels a copy of the original?
print('='*70)
print('LABEL CONSISTENCY: Clean_640 vs Original test_nopatch')
print('='*70)

mismatches = []

for bb_num in range(1, 10):
    bb = f'billboard0{bb_num}'
    orig_lab = Path(ORIG_ROOT)/bb/'labels/test_nopatch'
    clean_lab = Path(CLEAN_640)/f'labels/{bb}'
    orig_img  = Path(ORIG_ROOT)/bb/'images/test_nopatch'
    clean_img = Path(CLEAN_640)/f'images/{bb}'

    if not orig_lab.exists() or not clean_lab.exists():
        print(f'{bb}: missing folder — skip')
        continue

    # Match by stem
    orig_files  = {f.stem: f for f in orig_lab.iterdir()  if f.suffix == '.txt'}
    clean_files = {f.stem: f for f in clean_lab.iterdir() if f.suffix == '.txt'}

    # Check stems match
    only_orig  = set(orig_files) - set(clean_files)
    only_clean = set(clean_files) - set(orig_files)
    common     = set(orig_files) & set(clean_files)

    print(f'\n── {bb} ──')
    print(f'  Orig labels : {len(orig_files)}  Clean_640 labels: {len(clean_files)}')
    if only_orig:  print(f'  Only in orig : {list(only_orig)[:5]}')
    if only_clean: print(f'  Only in clean: {list(only_clean)[:5]}')

    box_diffs = 0
    content_diffs = []
    for stem in sorted(common):
        orig_lbls  = read_labels(orig_files[stem])
        clean_lbls = read_labels(clean_files[stem])
        if len(orig_lbls) != len(clean_lbls):
            box_diffs += 1
            content_diffs.append((stem, len(orig_lbls), len(clean_lbls)))
        else:
            # Check if content matches
            for o, c in zip(sorted(orig_lbls), sorted(clean_lbls)):
                if o[0] != c[0] or abs(o[1]-c[1])>0.01 or abs(o[2]-c[2])>0.01:
                    content_diffs.append((stem, 'content_mismatch', o, c))
                    break

    if not content_diffs:
        print(f'  ✅ Labels match perfectly')
    else:
        print(f'  ❌ {len(content_diffs)} mismatches found!')
        for d in content_diffs[:3]:
            print(f'     {d}')
        mismatches.append((bb, content_diffs))

print(f'\n{"="*70}')
if not mismatches:
    print('✅ All Clean_640 labels match originals perfectly')
else:
    print(f'❌ Mismatches in {len(mismatches)} billboards — Clean_640 labels may be wrong!')

In [ ]:
# ── CELL 7: Visual check — Original dataset ───────────────────
# Shows 3 random images per billboard with GT boxes overlaid

def draw_boxes(img_path, lab_path, title='', img_size=None):
    img = Image.open(img_path).convert('RGB')
    W, H = img.size
    if img_size:
        img = img.resize(img_size, Image.BILINEAR)
        W2, H2 = img_size
    else:
        W2, H2 = W, H
    draw = ImageDraw.Draw(img)
    labels = read_labels(lab_path)
    for (c,cx,cy,bw,bh) in labels:
        x1 = int((cx-bw/2)*W2); y1 = int((cy-bh/2)*H2)
        x2 = int((cx+bw/2)*W2); y2 = int((cy+bh/2)*H2)
        color = COLORS.get(c,'#FFFFFF')
        draw.rectangle([x1,y1,x2,y2], outline=color, width=3)
        draw.text((x1+2, y1+2), NAMES.get(c,str(c)), fill=color)
    return img, len(labels)

print('VISUAL CHECK — ORIGINAL dataset (3 images per billboard)')
fig, axes = plt.subplots(9, 3, figsize=(18, 54))

for bb_num in range(1, 10):
    bb = f'billboard0{bb_num}'
    img_dir = Path(ORIG_ROOT)/bb/'images/train'
    lab_dir = Path(ORIG_ROOT)/bb/'labels/train'
    if not img_dir.exists(): continue
    imgs = sorted([f for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXTS])
    samples = random.sample(imgs, min(3, len(imgs)))
    for col, img_path in enumerate(samples):
        lab_path = lab_dir / f'{img_path.stem}.txt'
        vis, n_boxes = draw_boxes(img_path, lab_path, img_size=(480,270))
        ax = axes[bb_num-1][col]
        ax.imshow(np.array(vis))
        ax.set_title(f'{bb} | {img_path.name} | {n_boxes} boxes', fontsize=8)
        ax.axis('off')

plt.suptitle('Original Dataset — GT Labels', fontsize=14, y=1.001)
plt.tight_layout()
plt.savefig('/content/orig_dataset_check.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved: /content/orig_dataset_check.png')

In [ ]:
# ── CELL 8: Visual check — Clean_640 ─────────────────────────
print('VISUAL CHECK — Clean_640 (3 images per billboard)')
fig, axes = plt.subplots(9, 3, figsize=(18, 54))

for bb_num in range(1, 10):
    bb = f'billboard0{bb_num}'
    img_dir = Path(CLEAN_640)/f'images/{bb}'
    lab_dir = Path(CLEAN_640)/f'labels/{bb}'
    if not img_dir.exists(): continue
    imgs = sorted([f for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXTS])
    samples = random.sample(imgs, min(3, len(imgs)))
    for col, img_path in enumerate(samples):
        lab_path = lab_dir / f'{img_path.stem}.txt'
        vis, n_boxes = draw_boxes(img_path, lab_path, img_size=(640,640))
        ax = axes[bb_num-1][col]
        ax.imshow(np.array(vis))
        ax.set_title(f'{bb} | {img_path.name} | {n_boxes} boxes', fontsize=8)
        ax.axis('off')

plt.suptitle('Clean_640 — GT Labels', fontsize=14, y=1.001)
plt.tight_layout()
plt.savefig('/content/clean640_check.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved: /content/clean640_check.png')

In [ ]:
# ── CELL 9: Visual check — Patched datasets ───────────────────
print('VISUAL CHECK — Patched datasets (3 images per billboard)')
print('GT labels taken from Clean_640 (same images, different pixels)')
fig, axes = plt.subplots(9, 3, figsize=(18, 54))

for bb_num in range(1, 10):
    img_dir = Path(PATCHED_ROOT)/f'patched_dataset{bb_num}/images/test'
    lab_dir = Path(CLEAN_640)/f'labels/billboard0{bb_num}'  # GT from Clean_640
    if not img_dir.exists(): continue
    imgs = sorted([f for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXTS])
    samples = random.sample(imgs, min(3, len(imgs)))
    for col, img_path in enumerate(samples):
        lab_path = lab_dir / f'{img_path.stem}.txt'
        vis, n_boxes = draw_boxes(img_path, lab_path, img_size=(640,640))
        ax = axes[bb_num-1][col]
        ax.imshow(np.array(vis))
        ax.set_title(f'patched_dataset{bb_num} | {img_path.name} | {n_boxes} GT boxes', fontsize=8)
        ax.axis('off')

plt.suptitle('Patched Datasets — GT Labels from Clean_640', fontsize=14, y=1.001)
plt.tight_layout()
plt.savefig('/content/patched_check.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved: /content/patched_check.png')

In [ ]:
# ── CELL 10: Side-by-side — same image, orig vs clean_640 label
# This is the smoking gun — if labels differ visually, Clean_640 is wrong
print('SIDE-BY-SIDE: Same image — Original label vs Clean_640 label')

fig, axes = plt.subplots(9, 4, figsize=(24, 54))

for bb_num in range(1, 10):
    bb = f'billboard0{bb_num}'
    orig_img_dir = Path(ORIG_ROOT)/bb/'images/test_nopatch'
    orig_lab_dir = Path(ORIG_ROOT)/bb/'labels/test_nopatch'
    clean_img_dir = Path(CLEAN_640)/f'images/{bb}'
    clean_lab_dir = Path(CLEAN_640)/f'labels/{bb}'

    if not orig_img_dir.exists() or not clean_img_dir.exists(): continue

    # Find images that exist in both
    orig_imgs  = {f.stem: f for f in orig_img_dir.iterdir() if f.suffix.lower() in IMG_EXTS}
    clean_imgs = {f.stem: f for f in clean_img_dir.iterdir() if f.suffix.lower() in IMG_EXTS}
    common     = sorted(set(orig_imgs) & set(clean_imgs))

    if not common:
        print(f'{bb}: no common stems between orig and clean_640')
        # Try first 2 of each
        orig_sample = sorted(orig_imgs.values())[:2]
        for col, img_path in enumerate(orig_sample[:2]):
            lab_path = orig_lab_dir / f'{img_path.stem}.txt'
            vis, n = draw_boxes(img_path, lab_path, img_size=(480,480))
            ax = axes[bb_num-1][col]
            ax.imshow(np.array(vis))
            ax.set_title(f'ORIG: {img_path.name}\n{n} boxes', fontsize=7)
            ax.axis('off')
        clean_sample = sorted(clean_imgs.values())[:2]
        for col, img_path in enumerate(clean_sample[:2]):
            lab_path = clean_lab_dir / f'{img_path.stem}.txt'
            vis, n = draw_boxes(img_path, lab_path, img_size=(480,480))
            ax = axes[bb_num-1][col+2]
            ax.imshow(np.array(vis))
            ax.set_title(f'CLEAN640: {img_path.name}\n{n} boxes', fontsize=7)
            ax.axis('off')
        continue

    samples = random.sample(common, min(2, len(common)))
    for col, stem in enumerate(samples):
        # Original
        orig_vis, orig_n = draw_boxes(
            orig_imgs[stem], orig_lab_dir/f'{stem}.txt', img_size=(480,480))
        ax = axes[bb_num-1][col*2]
        ax.imshow(np.array(orig_vis))
        ax.set_title(f'ORIG {bb}\n{stem[:20]} | {orig_n} boxes', fontsize=7, color='green')
        ax.axis('off')
        # Clean_640
        clean_vis, clean_n = draw_boxes(
            clean_imgs[stem], clean_lab_dir/f'{stem}.txt', img_size=(480,480))
        ax = axes[bb_num-1][col*2+1]
        ax.imshow(np.array(clean_vis))
        ax.set_title(f'CLEAN_640 {bb}\n{stem[:20]} | {clean_n} boxes', fontsize=7,
                     color='red' if clean_n != orig_n else 'blue')
        ax.axis('off')

plt.suptitle('Side-by-side: Original (green) vs Clean_640 (red=mismatch / blue=match)',
             fontsize=13, y=1.001)
plt.tight_layout()
plt.savefig('/content/comparison_orig_vs_clean640.png', dpi=80, bbox_inches='tight')
plt.show()
print('Saved: /content/comparison_orig_vs_clean640.png')
print('\nRed title = box count mismatch between orig and clean_640')
print('If boxes look misaligned visually → Clean_640 labels are wrong')

In [ ]:
# ── CELL 11: Summary verdict ──────────────────────────────────
print('='*70)
print('DATASET AUDIT SUMMARY')
print('='*70)

def summarise(stats_dict, name):
    total_imgs  = sum(s['n_images']        for s in stats_dict.values() if s)
    total_boxes = sum(s['n_boxes_total']   for s in stats_dict.values() if s)
    total_miss  = sum(s['n_labels_missing']for s in stats_dict.values() if s)
    total_empty = sum(s['n_empty_labels']  for s in stats_dict.values() if s)
    total_bad   = sum(len(s['issues'])     for s in stats_dict.values() if s)
    avg_boxes   = total_boxes / total_imgs if total_imgs > 0 else 0
    print(f'\n{name}')
    print(f'  Images        : {total_imgs}')
    print(f'  Total boxes   : {total_boxes}  avg: {avg_boxes:.2f}/img')
    print(f'  Missing labels: {total_miss}')
    print(f'  Empty labels  : {total_empty}')
    print(f'  Bad boxes     : {total_bad}')
    status = '✅ CLEAN' if total_bad == 0 and total_miss == 0 else '⚠️  ISSUES FOUND'
    print(f'  Status        : {status}')

summarise(orig_stats,   'ORIGINAL (2dod/2dod)')
summarise(clean_stats,  'CLEAN_640')
summarise(patch_stats,  'PATCHED DATASETS')

print(f'\n{"="*70}')
print('Saved images:')
print('  /content/orig_dataset_check.png      — original GT boxes')
print('  /content/clean640_check.png          — Clean_640 GT boxes')
print('  /content/patched_check.png           — patched images + Clean_640 GT')
print('  /content/comparison_orig_vs_clean640.png — side-by-side comparison')
print('\nIf clean640_check.png shows misaligned boxes → labels are wrong')
print('If comparison shows different box counts → Clean_640 labels diverged from original')